# 集群上lumpy运行流程

## 一、数据来源

### 197个样本路径：/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams/

### 80组配对样本名称列表：/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt

## 二、配置环境与运行 

In [2]:
conda create -n lumpy python=2.7 -y
conda activate lumpy

conda install -c bioconda lumpy-sv svtyper samtools bcftools samblaster -y

conda install -c conda-forge openssl=1.0 -y

# 输入下面命令看是否成功安装
lumpyexpress -h
svtyper -h

SyntaxError: invalid syntax (2371128687.py, line 1)

### 1、单组配对样本测试脚本/mnt/home/ygjx/chenkejin/Lumpy/test_single_lumpy.sh，代码如下：

In [ ]:
#!/bin/bash
#SBATCH --job-name=lumpy_single_pdac
#SBATCH --nodes=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=48G
#SBATCH --output=/mnt/home/ygjx/chenkejin/Lumpy/logs/test_single_lumpy_%j.out

set -euo pipefail

source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate lumpy_env

SAMTOOLS="${SAMTOOLS:-samtools}"
SVTYPER="${SVTYPER:-svtyper}"
EXTRACT_SCRIPT="${EXTRACT_SCRIPT:-/mnt/home/ygjx/chenkejin/anaconda3/envs/lumpy_env/share/lumpy-sv-0.2.13-0/scripts/extractSplitReads_BwaMem}"

THREADS="${THREADS:-${SLURM_CPUS_PER_TASK:-8}}"
MANIFEST="${MANIFEST:-/mnt/home/ygjx/chenkejin/bam_qc/PDAC_WGS_full_BAM_QC/pdac_197_bam_manifest.tsv}"
WORK_DIR="${WORK_DIR:-/mnt/home/ygjx/chenkejin/Lumpy}"

NORMAL_ID="1866277N"
TUMOR_ID="1866277T"
PREFIX=""

usage() {
  cat <<'EOF'
Usage:
  bash test_single_lumpy.sh [--normal NORMAL_ID] [--tumor TUMOR_ID] [--prefix PREFIX]

Defaults:
  --normal 1866277N
  --tumor  1866277T

Optional environment variables:
  MANIFEST   Default: /mnt/home/ygjx/chenkejin/bam_qc/PDAC_WGS_full_BAM_QC/pdac_197_bam_manifest.tsv
  WORK_DIR   Default: /mnt/home/ygjx/chenkejin/Lumpy
  THREADS    Default: SLURM_CPUS_PER_TASK or 8
EOF
}

while [ "$#" -gt 0 ]; do
  case "$1" in
    --normal)
      NORMAL_ID="${2:?ERROR: --normal needs a sample id}"
      shift 2
      ;;
    --tumor)
      TUMOR_ID="${2:?ERROR: --tumor needs a sample id}"
      shift 2
      ;;
    --prefix)
      PREFIX="${2:?ERROR: --prefix needs a value}"
      shift 2
      ;;
    --manifest)
      MANIFEST="${2:?ERROR: --manifest needs a path}"
      shift 2
      ;;
    --work-dir)
      WORK_DIR="${2:?ERROR: --work-dir needs a path}"
      shift 2
      ;;
    -h|--help)
      usage
      exit 0
      ;;
    *)
      echo "ERROR: unknown argument: $1" >&2
      usage >&2
      exit 1
      ;;
  esac
done

if [ -z "${PREFIX}" ]; then
  PREFIX="${NORMAL_ID%N}"
fi

mkdir -p "${WORK_DIR}/logs"
SAMPLE_LOG="${WORK_DIR}/logs/${PREFIX}.test_single_lumpy.log"
exec > >(tee -i "${SAMPLE_LOG}") 2>&1

echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] LUMPY single test: ${PREFIX}"
echo "Normal: ${NORMAL_ID}"
echo "Tumor : ${TUMOR_ID}"
echo "Manifest: ${MANIFEST}"
echo "Threads: ${THREADS}"
echo "Somatic filter: Normal AO=0, Tumor AB>=0.02, Tumor SU>=8"
echo "Chromosome cleanup: chr1-22, chrX, chrY"
echo "=========================================================="

if [ ! -f "${MANIFEST}" ]; then
  echo "ERROR: manifest not found: ${MANIFEST}" >&2
  echo "Run the PDAC BAM manifest script first." >&2
  exit 1
fi

if [ ! -x "${EXTRACT_SCRIPT}" ]; then
  echo "ERROR: extractSplitReads_BwaMem not executable: ${EXTRACT_SCRIPT}" >&2
  exit 1
fi

get_manifest_field() {
  local sample="$1"
  local field="$2"
  awk -F'\t' -v sample="${sample}" -v field="${field}" '
    NR==1 {
      for (i=1; i<=NF; i++) col[$i]=i
      if (!("sample" in col) || !(field in col)) exit 2
      next
    }
    $(col["sample"]) == sample {
      print $(col[field])
      found=1
      exit
    }
    END {
      if (!found) exit 1
    }
  ' "${MANIFEST}"
}

resolve_bai() {
  local bam="$1"
  local manifest_bai="${2:-NA}"

  if [ "${manifest_bai}" != "NA" ] && [ -f "${manifest_bai}" ]; then
    echo "${manifest_bai}"
  elif [ -f "${bam}.bai" ]; then
    echo "${bam}.bai"
  elif [ -f "${bam%.bam}.bai" ]; then
    echo "${bam%.bam}.bai"
  elif [ -f "${bam}.csi" ]; then
    echo "${bam}.csi"
  elif [ -f "${bam%.bam}.csi" ]; then
    echo "${bam%.bam}.csi"
  else
    echo "NOT_FOUND"
  fi
}

NORMAL_BAM_SRC="$(get_manifest_field "${NORMAL_ID}" "bam")"
TUMOR_BAM_SRC="$(get_manifest_field "${TUMOR_ID}" "bam")"
NORMAL_BAI_SRC="$(resolve_bai "${NORMAL_BAM_SRC}" "$(get_manifest_field "${NORMAL_ID}" "bai" || echo NA)")"
TUMOR_BAI_SRC="$(resolve_bai "${TUMOR_BAM_SRC}" "$(get_manifest_field "${TUMOR_ID}" "bai" || echo NA)")"

if [ ! -f "${NORMAL_BAM_SRC}" ] || [ ! -f "${TUMOR_BAM_SRC}" ]; then
  echo "ERROR: BAM missing after manifest lookup." >&2
  echo "Normal BAM: ${NORMAL_BAM_SRC}" >&2
  echo "Tumor BAM : ${TUMOR_BAM_SRC}" >&2
  exit 1
fi

SANDBOX="${WORK_DIR}/sandbox/${PREFIX}"
INPUT_DIR="${SANDBOX}/inputs"
SPLITTER_DIR="${SANDBOX}/splitters"
TMP_DIR="${SANDBOX}/tmp"
FINAL_VCF_DIR="${WORK_DIR}/PDAC_197_single_test_result"

mkdir -p "${INPUT_DIR}" "${SPLITTER_DIR}" "${TMP_DIR}" "${FINAL_VCF_DIR}"

NORMAL_BAM="${INPUT_DIR}/$(basename "${NORMAL_BAM_SRC}")"
TUMOR_BAM="${INPUT_DIR}/$(basename "${TUMOR_BAM_SRC}")"

ln -sf "${NORMAL_BAM_SRC}" "${NORMAL_BAM}"
ln -sf "${TUMOR_BAM_SRC}" "${TUMOR_BAM}"

if [ "${NORMAL_BAI_SRC}" != "NOT_FOUND" ]; then
  ln -sf "${NORMAL_BAI_SRC}" "${NORMAL_BAM}.bai"
fi
if [ "${TUMOR_BAI_SRC}" != "NOT_FOUND" ]; then
  ln -sf "${TUMOR_BAI_SRC}" "${TUMOR_BAM}.bai"
fi

echo "Normal BAM source: ${NORMAL_BAM_SRC}"
echo "Tumor  BAM source: ${TUMOR_BAM_SRC}"
echo "Normal BAI source: ${NORMAL_BAI_SRC}"
echo "Tumor  BAI source: ${TUMOR_BAI_SRC}"

echo "[$(date)] Checking BAM integrity..."
"${SAMTOOLS}" quickcheck -v "${NORMAL_BAM}" "${TUMOR_BAM}"

tumor_splitters_bam="${SPLITTER_DIR}/${PREFIX}.tumor.splitters.bam"
normal_splitters_bam="${SPLITTER_DIR}/${PREFIX}.normal.splitters.bam"
tumor_discordants_bam="${SPLITTER_DIR}/${PREFIX}.tumor.discordants.bam"
normal_discordants_bam="${SPLITTER_DIR}/${PREFIX}.normal.discordants.bam"

tumor_normal_vcf="${SANDBOX}/${PREFIX}.tumor_normal.lumpy.vcf"
lumpy_genotyped_vcf="${SANDBOX}/${PREFIX}.lumpy.genotyped.vcf"
vcf_output="${SANDBOX}/${PREFIX}.lumpy.somatic.vcf"
vcf_filter_output="${FINAL_VCF_DIR}/${PREFIX}.lumpy.somatic.filtered.vcf"

trap 'echo "[$(date)] Cleaning large intermediate BAM files..."; rm -f "$tumor_splitters_bam" "$normal_splitters_bam" "$tumor_discordants_bam" "$normal_discordants_bam"' EXIT INT TERM

{
  echo "[$(date)] Step 1: Extract splitters, whole genome..."
  "${SAMTOOLS}" view -@ "${THREADS}" -h "${TUMOR_BAM}" | "${EXTRACT_SCRIPT}" -i stdin | "${SAMTOOLS}" view -@ "${THREADS}" -Sb - > "${tumor_splitters_bam}"
  "${SAMTOOLS}" view -@ "${THREADS}" -h "${NORMAL_BAM}" | "${EXTRACT_SCRIPT}" -i stdin | "${SAMTOOLS}" view -@ "${THREADS}" -Sb - > "${normal_splitters_bam}"

  echo "[$(date)] Step 2: Extract discordants, whole genome..."
  "${SAMTOOLS}" view -@ "${THREADS}" -b -F 1294 "${TUMOR_BAM}" > "${tumor_discordants_bam}"
  "${SAMTOOLS}" view -@ "${THREADS}" -b -F 1294 "${NORMAL_BAM}" > "${normal_discordants_bam}"

  echo "[$(date)] Step 3: Run lumpyexpress..."
  lumpyexpress \
    -B "${TUMOR_BAM}","${NORMAL_BAM}" \
    -S "${tumor_splitters_bam}","${normal_splitters_bam}" \
    -D "${tumor_discordants_bam}","${normal_discordants_bam}" \
    -o "${tumor_normal_vcf}" \
    -T "${TMP_DIR}"

  echo "[$(date)] Step 4: Run svtyper..."
  "${SVTYPER}" \
    -B "${TUMOR_BAM}","${NORMAL_BAM}" \
    -i "${tumor_normal_vcf}" \
    -o "${lumpy_genotyped_vcf}"

  echo "[$(date)] Step 5: Somatic filtering by FORMAT tags..."
  awk -F'\t' -v AB=0.02 -v SU=8 '
    BEGIN {OFS="\t"}
    /^#/ {print; next}
    {
      n=split($9,f,":")
      split("", idx)
      for (i=1; i<=n; i++) idx[f[i]]=i

      split($10,t,":")
      split($11,nm,":")

      ao=(idx["AO"] ? nm[idx["AO"]] : ".")
      ab=(idx["AB"] ? t[idx["AB"]] : ".")
      su=(idx["SU"] ? t[idx["SU"]] : ".")

      if (ao==".") ao=0
      if (ab==".") ab=0
      if (su==".") su=0

      if (ao+0==0 && ab+0>=AB && su+0>=SU) print
    }
  ' "${lumpy_genotyped_vcf}" > "${vcf_output}"

  echo "[$(date)] Step 6: Clean non-standard chromosomes..."
  awk '/^#/ || $1 ~ /^chr([1-9]|1[0-9]|2[0-2]|X|Y)$/' "${vcf_output}" > "${vcf_filter_output}"

  echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] ${PREFIX}"
  echo "Final VCF: ${vcf_filter_output}"
}


### bash test_single_lumpy.sh --normal 1866277N --tumor 1866277T --prefix 1866277   单样本测试

### 2、集群批量处理脚本/mnt/home/ygjx/chenkejin/Lumpy/batch_lumpy.sh，代码如下：

In [ ]:
#!/bin/bash
#SBATCH --job-name=Lumpy_PDAC197
#SBATCH --nodes=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=48G
#SBATCH --output=/mnt/home/ygjx/chenkejin/Lumpy/logs/slurm_array_%A_%a.out
#SBATCH --error=/mnt/home/ygjx/chenkejin/Lumpy/logs/slurm_array_%A_%a.err

set -euo pipefail

source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate lumpy_env

SAMTOOLS="${SAMTOOLS:-samtools}"
SVTYPER="${SVTYPER:-svtyper}"
EXTRACT_SCRIPT="${EXTRACT_SCRIPT:-/mnt/home/ygjx/chenkejin/anaconda3/envs/lumpy_env/share/lumpy-sv-0.2.13-0/scripts/extractSplitReads_BwaMem}"

THREADS="${THREADS:-${SLURM_CPUS_PER_TASK:-8}}"
MANIFEST="${MANIFEST:-/mnt/home/ygjx/chenkejin/bam_qc/PDAC_WGS_full_BAM_QC/pdac_197_bam_manifest.tsv}"
TASK_LIST="${TASK_LIST:-/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt}"
WORK_DIR="${WORK_DIR:-/mnt/home/ygjx/chenkejin/Lumpy}"
MASTER_LOG="${WORK_DIR}/master_progress_pdac197.log"

if [ -z "${SLURM_ARRAY_TASK_ID:-}" ]; then
  echo "ERROR: this script must be submitted as a Slurm array job." >&2
  echo "Example:" >&2
  echo "  N=\\$(awk 'NF>=2 && \\$1 !~ /^#/{n++} END{print n+0}' ${TASK_LIST})" >&2
  echo "  sbatch --array=1-\\${N}%10 batch_lumpy.sh" >&2
  exit 1
fi

if [ ! -f "${MANIFEST}" ]; then
  echo "ERROR: manifest not found: ${MANIFEST}" >&2
  echo "Run the PDAC BAM manifest script first." >&2
  exit 1
fi

if [ ! -f "${TASK_LIST}" ]; then
  echo "ERROR: task list not found: ${TASK_LIST}" >&2
  echo "Expected format per non-comment line: NORMAL_ID<TAB_or_SPACE>TUMOR_ID" >&2
  exit 1
fi

if [ ! -x "${EXTRACT_SCRIPT}" ]; then
  echo "ERROR: extractSplitReads_BwaMem not executable: ${EXTRACT_SCRIPT}" >&2
  exit 1
fi

TOTAL_TASKS="$(awk 'NF>=2 && $1 !~ /^#/{n++} END{print n+0}' "${TASK_LIST}")"
LINE="$(awk -v idx="${SLURM_ARRAY_TASK_ID}" '
  NF>=2 && $1 !~ /^#/ {
    n++
    if (n==idx) {
      print $1 "\t" $2
      exit
    }
  }
' "${TASK_LIST}" | tr -d '\r')"

if [ -z "${LINE}" ]; then
  echo "ERROR: empty task line for SLURM_ARRAY_TASK_ID=${SLURM_ARRAY_TASK_ID}" >&2
  echo "Total usable tasks in ${TASK_LIST}: ${TOTAL_TASKS}" >&2
  exit 1
fi

IFS=$'\t' read -r NORMAL_ID TUMOR_ID <<< "${LINE}"
PREFIX="${NORMAL_ID%N}"

mkdir -p "${WORK_DIR}/logs"
SAMPLE_LOG="${WORK_DIR}/logs/${PREFIX}.lumpy.log"
exec > >(tee -i "${SAMPLE_LOG}") 2>&1

echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] LUMPY PDAC sample pair: ${PREFIX}"
echo "Task: ${SLURM_ARRAY_TASK_ID}/${TOTAL_TASKS}"
echo "Node: $(hostname)"
echo "Normal: ${NORMAL_ID}"
echo "Tumor : ${TUMOR_ID}"
echo "Manifest: ${MANIFEST}"
echo "Task list: ${TASK_LIST}"
echo "Threads: ${THREADS}"
echo "Somatic filter: Normal AO=0, Tumor AB>=0.02, Tumor SU>=8"
echo "Chromosome cleanup: chr1-22, chrX, chrY"
echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] Task ${SLURM_ARRAY_TASK_ID}/${TOTAL_TASKS}: ${PREFIX}" >> "${MASTER_LOG}"

get_manifest_field() {
  local sample="$1"
  local field="$2"
  awk -F'\t' -v sample="${sample}" -v field="${field}" '
    NR==1 {
      for (i=1; i<=NF; i++) col[$i]=i
      if (!("sample" in col) || !(field in col)) exit 2
      next
    }
    $(col["sample"]) == sample {
      print $(col[field])
      found=1
      exit
    }
    END {
      if (!found) exit 1
    }
  ' "${MANIFEST}"
}

resolve_bai() {
  local bam="$1"
  local manifest_bai="${2:-NA}"

  if [ "${manifest_bai}" != "NA" ] && [ -f "${manifest_bai}" ]; then
    echo "${manifest_bai}"
  elif [ -f "${bam}.bai" ]; then
    echo "${bam}.bai"
  elif [ -f "${bam%.bam}.bai" ]; then
    echo "${bam%.bam}.bai"
  elif [ -f "${bam}.csi" ]; then
    echo "${bam}.csi"
  elif [ -f "${bam%.bam}.csi" ]; then
    echo "${bam%.bam}.csi"
  else
    echo "NOT_FOUND"
  fi
}

NORMAL_BAM_SRC="$(get_manifest_field "${NORMAL_ID}" "bam")"
TUMOR_BAM_SRC="$(get_manifest_field "${TUMOR_ID}" "bam")"
NORMAL_BAI_SRC="$(resolve_bai "${NORMAL_BAM_SRC}" "$(get_manifest_field "${NORMAL_ID}" "bai" || echo NA)")"
TUMOR_BAI_SRC="$(resolve_bai "${TUMOR_BAM_SRC}" "$(get_manifest_field "${TUMOR_ID}" "bai" || echo NA)")"

if [ ! -f "${NORMAL_BAM_SRC}" ] || [ ! -f "${TUMOR_BAM_SRC}" ]; then
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] BAM missing after manifest lookup."
  echo "Normal BAM: ${NORMAL_BAM_SRC}"
  echo "Tumor  BAM: ${TUMOR_BAM_SRC}"
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] Task ${SLURM_ARRAY_TASK_ID}/${TOTAL_TASKS}: ${PREFIX} missing BAM" >> "${MASTER_LOG}"
  exit 1
fi

SANDBOX="${WORK_DIR}/sandbox/${PREFIX}"
INPUT_DIR="${SANDBOX}/inputs"
SPLITTER_DIR="${SANDBOX}/splitters"
TMP_DIR="${SANDBOX}/tmp"
FINAL_VCF_DIR="${WORK_DIR}/PDAC_197_result"

mkdir -p "${INPUT_DIR}" "${SPLITTER_DIR}" "${TMP_DIR}" "${FINAL_VCF_DIR}"

NORMAL_BAM="${INPUT_DIR}/$(basename "${NORMAL_BAM_SRC}")"
TUMOR_BAM="${INPUT_DIR}/$(basename "${TUMOR_BAM_SRC}")"

ln -sf "${NORMAL_BAM_SRC}" "${NORMAL_BAM}"
ln -sf "${TUMOR_BAM_SRC}" "${TUMOR_BAM}"

if [ "${NORMAL_BAI_SRC}" != "NOT_FOUND" ]; then
  ln -sf "${NORMAL_BAI_SRC}" "${NORMAL_BAM}.bai"
fi
if [ "${TUMOR_BAI_SRC}" != "NOT_FOUND" ]; then
  ln -sf "${TUMOR_BAI_SRC}" "${TUMOR_BAM}.bai"
fi

echo "Normal BAM source: ${NORMAL_BAM_SRC}"
echo "Tumor  BAM source: ${TUMOR_BAM_SRC}"
echo "Normal BAI source: ${NORMAL_BAI_SRC}"
echo "Tumor  BAI source: ${TUMOR_BAI_SRC}"

echo "[$(date)] Checking BAM integrity..."
"${SAMTOOLS}" quickcheck -v "${NORMAL_BAM}" "${TUMOR_BAM}"

tumor_splitters_bam="${SPLITTER_DIR}/${PREFIX}.tumor.splitters.bam"
normal_splitters_bam="${SPLITTER_DIR}/${PREFIX}.normal.splitters.bam"
tumor_discordants_bam="${SPLITTER_DIR}/${PREFIX}.tumor.discordants.bam"
normal_discordants_bam="${SPLITTER_DIR}/${PREFIX}.normal.discordants.bam"

tumor_normal_vcf="${SANDBOX}/${PREFIX}.tumor_normal.lumpy.vcf"
lumpy_genotyped_vcf="${SANDBOX}/${PREFIX}.lumpy.genotyped.vcf"
vcf_output="${SANDBOX}/${PREFIX}.lumpy.somatic.vcf"
vcf_filter_output="${FINAL_VCF_DIR}/${PREFIX}.lumpy.somatic.filtered.vcf"

trap 'echo "[$(date)] Cleaning large intermediate BAM files..."; rm -f "$tumor_splitters_bam" "$normal_splitters_bam" "$tumor_discordants_bam" "$normal_discordants_bam"' EXIT INT TERM

{
  echo "[$(date)] Step 1: Extract splitters, whole genome..."
  "${SAMTOOLS}" view -@ "${THREADS}" -h "${TUMOR_BAM}" | "${EXTRACT_SCRIPT}" -i stdin | "${SAMTOOLS}" view -@ "${THREADS}" -Sb - > "${tumor_splitters_bam}"
  "${SAMTOOLS}" view -@ "${THREADS}" -h "${NORMAL_BAM}" | "${EXTRACT_SCRIPT}" -i stdin | "${SAMTOOLS}" view -@ "${THREADS}" -Sb - > "${normal_splitters_bam}"

  echo "[$(date)] Step 2: Extract discordants, whole genome..."
  "${SAMTOOLS}" view -@ "${THREADS}" -b -F 1294 "${TUMOR_BAM}" > "${tumor_discordants_bam}"
  "${SAMTOOLS}" view -@ "${THREADS}" -b -F 1294 "${NORMAL_BAM}" > "${normal_discordants_bam}"

  echo "[$(date)] Step 3: Run lumpyexpress..."
  lumpyexpress \
    -B "${TUMOR_BAM}","${NORMAL_BAM}" \
    -S "${tumor_splitters_bam}","${normal_splitters_bam}" \
    -D "${tumor_discordants_bam}","${normal_discordants_bam}" \
    -o "${tumor_normal_vcf}" \
    -T "${TMP_DIR}"

  echo "[$(date)] Step 4: Run svtyper..."
  "${SVTYPER}" \
    -B "${TUMOR_BAM}","${NORMAL_BAM}" \
    -i "${tumor_normal_vcf}" \
    -o "${lumpy_genotyped_vcf}"

  echo "[$(date)] Step 5: Somatic filtering by FORMAT tags..."
  awk -F'\t' -v AB=0.02 -v SU=8 '
    BEGIN {OFS="\t"}
    /^#/ {print; next}
    {
      n=split($9,f,":")
      split("", idx)
      for (i=1; i<=n; i++) idx[f[i]]=i

      split($10,t,":")
      split($11,nm,":")

      ao=(idx["AO"] ? nm[idx["AO"]] : ".")
      ab=(idx["AB"] ? t[idx["AB"]] : ".")
      su=(idx["SU"] ? t[idx["SU"]] : ".")

      if (ao==".") ao=0
      if (ab==".") ab=0
      if (su==".") su=0

      if (ao+0==0 && ab+0>=AB && su+0>=SU) print
    }
  ' "${lumpy_genotyped_vcf}" > "${vcf_output}"

  echo "[$(date)] Step 6: Clean non-standard chromosomes..."
  awk '/^#/ || $1 ~ /^chr([1-9]|1[0-9]|2[0-2]|X|Y)$/' "${vcf_output}" > "${vcf_filter_output}"

  echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] ${PREFIX}"
  echo "Final VCF: ${vcf_filter_output}"
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] Task ${SLURM_ARRAY_TASK_ID}/${TOTAL_TASKS}: ${PREFIX}" >> "${MASTER_LOG}"
} || {
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] ${PREFIX} failed"
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] Task ${SLURM_ARRAY_TASK_ID}/${TOTAL_TASKS}: ${PREFIX} failed" >> "${MASTER_LOG}"
  exit 1
}


### 批量提交方式：

In [ ]:
TASK_LIST="/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt"
N=$(awk 'NF>=2 && $1 !~ /^#/{n++} END{print n+0}' "${TASK_LIST}")

sbatch --array=1-${N}%30 batch_lumpy.sh

## 结果文件路径：/mnt/home/ygjx/chenkejin/Lumpy/PDAC_197_result